# Boiler Plate Code for All Projects


In [96]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
import operator
from typing import TypedDict, List, Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from dotenv import load_dotenv
import os
load_dotenv()


if os.environ['GOOGLE_API_KEY']:
    print("Google API Key is set.")
else:
    raise ValueError("Google API Key is not set.")

Google API Key is set.


## Set the LLM Object


In [97]:
llm =ChatGoogleGenerativeAI(model="gemini-2.5-flash")

## Pydantic Schema for Section and Sections


In [98]:
# ── Pydantic Schemas ──────────────────────────────────────────────────────
class Section(BaseModel):
    name: str = Field(description="Name for this section of the report.")
    description: str = Field(description="Brief overview of the main topics.")

class Sections(BaseModel):
    sections: List[Section] = Field(description="Sections of the report.")

# Augmented LLM


In [110]:
planner = llm.with_structured_output(Sections)
planner.invoke("What is alpha?").sections

[Section(name='Definition of Alpha', description="In finance, alpha (α) is a measure of an investment's performance relative to a benchmark index. It represents the excess return an investment earns above the return of a comparable market index, indicating the value added by an active manager's skill or strategy.")]

# Graph States


In [100]:
class State(TypedDict):
    topic: str
    sections: List[Section]
    completed_sections: Annotated[list, operator.add]
    final_report: str


class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, operator.add]


# Nodes


In [101]:
def orchestrator(state: State) -> State:
    """Orchestrator that generates a plan for the report"""
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Generate a plan for the report. The report should be divided into sections, and each section should have a name and a brief description of the main topics that will be covered in that section. The report should be based on the following topic: {topic}. The report should be comprehensive and cover all relevant aspects of the topic."),
        ("human", "Here is the report topic: {topic}. Please generate a plan for the report, including the sections and their descriptions.")
    ])
    
    chain = prompt | planner
    response = chain.invoke({"topic": state['topic']})
    
    return {
        "sections": response.sections
    }

In [102]:
def llm_call(state: WorkerState):
    """LLM node that generates content for a section of the report"""
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"You are an expert writer tasked with generating content for a section of a report. The section is about {state['section'].name} and should cover the following topics: {state['section'].description}. Please generate comprehensive and well-structured content for this section."),
        ("human", f"Here is the section information: {state['section']}. Please generate content for this section of the report.")
    ])
    
    chain = prompt | llm
    response = chain.invoke({"section": state['section']})
    
    return {"completed_sections": [response.content]}


# Summarizer Node

In [103]:
def synthesizer(state: State):
    """Synthesize full report from sections"""
    completed_sections = state["completed_sections"]
    final_report = "\n\n---\n\n".join(completed_sections)
    return {"final_report": final_report}

# Worker Assigner Send Method

In [104]:
# ── Conditional Edge ──────────────────────────────────────────────────────
def assign_workers(state: State):
    """Assign a worker to each section in the plan"""
    return [Send("llm_call", {"section": s}) for s in state["sections"]]

In [105]:
# ── Build Graph ───────────────────────────────────────────────────────────
orchestrator_worker_builder = StateGraph(State)

orchestrator_worker_builder.add_node("orchestrator", orchestrator)
orchestrator_worker_builder.add_node("llm_call", llm_call)
orchestrator_worker_builder.add_node("synthesizer", synthesizer)

orchestrator_worker_builder.add_edge(START, "orchestrator")
orchestrator_worker_builder.add_conditional_edges(
    "orchestrator",
    assign_workers,
    ["llm_call"]
)
orchestrator_worker_builder.add_edge("llm_call", "synthesizer")
orchestrator_worker_builder.add_edge("synthesizer", END)

orchestrator_worker = orchestrator_worker_builder.compile()


# ── Run ───────────────────────────────────────────────────────────────────
state = orchestrator_worker.invoke({"topic": "LLM Scaling Laws"})
state

{'topic': 'LLM Scaling Laws',
 'sections': [Section(name='Introduction to LLM Scaling Laws', description='Define LLM Scaling Laws, explain their fundamental importance in the development and understanding of large language models, and outline the scope of the report.'),
  Section(name='Foundational Concepts and Early Discoveries', description='Discuss the theoretical underpinnings and initial empirical observations that led to the recognition of scaling phenomena in deep learning, highlighting key early research and methodologies.'),
  Section(name='Key Scaling Laws and Their Formulations', description='Detail the most prominent scaling laws, including their mathematical expressions and the relationships between model size, dataset size, training compute, and resulting performance. This section will cover findings such as those from Kaplan et al. and the Chinchilla optimal scaling laws.'),
  Section(name='Factors Influencing Scaling Performance', description='Explore the critical resou